In [1]:
import pandas as pd
import numpy as np
import regex
import random
from tqdm import tqdm
import pickle
import os

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import TreebankWordTokenizer
from nltk.stem import PorterStemmer
ps = PorterStemmer()

# Supervised text classification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn import metrics
from sklearn.model_selection import train_test_split

import gensim

from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
def short_classification_report(y_test, y_pred):
    print("    \tPrecision\tRecall")
    for label in set(y_pred):
        pr = metrics.precision_score(y_test, y_pred, pos_label=label)
        re = metrics.recall_score(y_test, y_pred, pos_label=label)
        print(f"{label}:\t{pr:0.2f}\t\t{re:0.2f}")
    ac = metrics.accuracy_score(y_test,y_pred)
    print(f"Accuracy: {ac:0.2f}")

class MyTokenizer:
    def tokenize(self, text, stem=True, to_lowercase=True):
        tokenizer = TreebankWordTokenizer()
        result = []
        word = r"\p{letter}"
        for sent in nltk.sent_tokenize(text):
            tokens = tokenizer.tokenize(sent)
            if stem:
                tokens = [ps.stem(t, to_lowercase=to_lowercase) for t in tokens if regex.search(word, t)]
            else:
                if to_lowercase:
                    tokens = [t.lower() for t in tokens if regex.search(word, t)]
                else:
                    tokens = [t for t in tokens if regex.search(word, t)]
            result += tokens
        return result

def collocations_phraser(data, tokenizer, min_count=10, min_score=0.5, show_df=True):
    tokens = [tokenizer(t) for t in data]
   
    phrases_model = gensim.models.Phrases(tokens, min_count=min_count, scoring="npmi", threshold=min_score)
    phraser = gensim.models.phrases.Phraser(phrases_model)
    tokens_phrases = [phraser[doc] for doc in tokens]
    
    if show_df:
        score_dict = phrases_model.export_phrases()
        scores = pd.DataFrame(score_dict.items(), columns=["phrase", "score"])
        display(HTML(scores.sort_values("score", ascending=False).to_html()))

    return tokens_phrases

mytokenizer = MyTokenizer()

### Load dataset that will be coded later

In [ ]:
min_word_count = 160

try:
    speech_data = pickle.load(open("/home/til/Documents/python/powi/extract-speeches-from-bundestag-minutes/data_processed/pp20.p", "rb"))
except:
    speech_data = pickle.load(open("data/pp20.p", "rb"))
speech_data = speech_data[speech_data.text.str.split().str.len().values >= min_word_count]
speech_data = speech_data[~speech_data.speech_id.duplicated()]

### Train the model

In [5]:
df = pd.read_csv("data/pp20_random_speeches_final_coding.csv")

df.loc[df.topic == "Außenhandel", "topic"] = "Makroökonomie"

In [7]:
X = df.text
y = df.topic

text_train, text_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    stratify=y,
                                                    random_state=42)

In [ ]:
load_data = False
save_model_params = False
iterations = 50
use_collocations = True
use_stemming = True
to_lowercase = True

mystopwords = stopwords.words('german')
if use_stemming:
    stemmed_stopwords = [ps.stem(sw) for sw in mystopwords]
    mystopwords = list(set(mystopwords + stemmed_stopwords))

SVC_grid = {
    "ngram_range": [(1, 1), (1, 2), (1, 3)],
    "max_df": np.linspace(0.5,1,9),
    "min_df": np.linspace(0, 0.1, 11), # np.arange(1,10,1),
    "classifier_C": np.linspace(1,100,20),
    "min_score": np.linspace(0.7,0.95,11)
}

best_params = {
    "ngram_range": None,
    "max_df": None,
    "min_df": None,
    "classifier_C": None,
    "min_score": None
}

recent_accuracy = 0

if not load_data:
    for i in tqdm(range(iterations)):
        ngram_range = random.choice(SVC_grid["ngram_range"])
        max_df = random.choice(SVC_grid["max_df"])
        min_df = random.choice(SVC_grid["min_df"])
        C = random.choice(SVC_grid["classifier_C"])

        if use_collocations:
            min_score = random.choice(SVC_grid["min_score"])
            tokens_phrases_train = collocations_phraser(data=text_train, 
                                                        tokenizer=lambda x: mytokenizer.tokenize(x, stem=use_stemming, to_lowercase=to_lowercase), 
                                                        min_score=min_score,
                                                        show_df=False)
            tokens_phrases_test = collocations_phraser(data=text_test, 
                                                       tokenizer=lambda x: mytokenizer.tokenize(x, stem=use_stemming, to_lowercase=to_lowercase), 
                                                       min_score=min_score,
                                                       show_df=False)
            tokenizer = lambda x: x
            training_data = tokens_phrases_train
            test_data = tokens_phrases_test
        else:
            tokenizer = lambda x: mytokenizer.tokenize(x, stem=use_stemming, to_lowercase=to_lowercase)
            training_data = text_train
            test_data = text_test

        vectorizer = TfidfVectorizer(
            stop_words=mystopwords,
            tokenizer=tokenizer,
            ngram_range=ngram_range,
            max_df=max_df,
            min_df=min_df,
            lowercase=False # Das hier ist nur False gesetzt damit der Vectorizer, sowohl mit einer Tokenliste als auch mit reinem Text arbeiten kann. 
                            # Ob ich die Daten in lowercase mache hängt allein von to_lowercase ab.
        )

        classifier = LinearSVC(
            C=C
        )
        
        X_train = vectorizer.fit_transform(training_data)
        X_test = vectorizer.transform(test_data)

        classifier.fit(X_train, y_train)
    
        y_pred = classifier.predict(X_test)

        accuracy = metrics.accuracy_score(y_test,y_pred)
        if accuracy > recent_accuracy:
            best_params["ngram_range"] = ngram_range
            best_params["max_df"] = max_df
            best_params["min_df"] = min_df
            best_params["classifier_C"] = C
            best_params["min_score"] = min_score
            rep = metrics.classification_report(y_test, y_pred)
            recent_accuracy = accuracy
    
    if save_model_params:
        if not os.path.isdir("model/"):
            os.mkdir("model/")
        with open("model/best_params.pkl", "wb") as p:
            pickle.dump(best_params, p)
else:
    with open("model/best_params.pkl", "rb") as p:
        best_params = pickle.load(p)
    
    ngram_range = best_params["ngram_range"]
    max_df = best_params["max_df"]
    min_df = best_params["min_df"]
    C = best_params["classifier_C"]
    min_score = best_params["min_score"]

    if min_score:
        tokens_phrases_train = collocations_phraser(data=text_train, 
                                                    tokenizer=lambda x: mytokenizer.tokenize(x, stem=use_stemming, to_lowercase=to_lowercase), 
                                                    min_score=min_score,
                                                    show_df=False)
        tokens_phrases_test = collocations_phraser(data=text_test, 
                                                   tokenizer=lambda x: mytokenizer.tokenize(x, stem=use_stemming, to_lowercase=to_lowercase), 
                                                   min_score=min_score,
                                                   show_df=False)
        tokenizer = lambda x: x
        training_data = tokens_phrases_train
        test_data = tokens_phrases_test
        lowercase = False
    else:
        tokenizer = lambda x: mytokenizer.tokenize(x, stem=use_stemming, to_lowercase=to_lowercase)
        training_data = text_train
        test_data = text_test
        lowercase = True

    vectorizer = TfidfVectorizer(
        stop_words=mystopwords,
        tokenizer=tokenizer,
        ngram_range=ngram_range,
        max_df=max_df,
        min_df=min_df,
        lowercase=lowercase
    )

    classifier = LinearSVC(
        C=C
    )
    
    X_train = vectorizer.fit_transform(training_data)
    X_test = vectorizer.transform(test_data)

    classifier.fit(X_train, y_train)

    y_pred = classifier.predict(X_test)

    rep = metrics.classification_report(y_test, y_pred)

print(best_params)
print(rep)

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [10:15<00:00, 12.32s/it]

{'ngram_range': (1, 1), 'max_df': np.float64(0.75), 'min_df': np.float64(0.0), 'classifier_C': np.float64(84.36842105263158), 'min_score': np.float64(0.825)}
                                                 precision    recall  f1-score   support

                                         Arbeit       0.89      0.73      0.80        11
                                        Bildung       0.75      0.43      0.55         7
                                   Bürgerrechte       0.67      0.55      0.60        11
                                        Energie       0.77      0.89      0.83        19
                                     Gesundheit       0.94      1.00      0.97        15
                                  Infrastruktur       0.86      0.75      0.80         8
                                   Innenpolitik       0.80      0.80      0.80        10
                     Internationale Beziehungen       0.57      0.80      0.67        15
                Kultur und Gesellschafts

### Code the data left

In [8]:
df_drop = df.drop(["document_name","date","top_id","drucksachen","context","speaker_id","text"],axis=1)
data_merged = speech_data.merge(df_drop,how="outer",on="speech_id")
df_coded = data_merged[~data_merged.topic.isnull()]
data_to_code = data_merged[data_merged.topic.isnull()]

In [ ]:
save = False

if use_collocations:
    tokens = collocations_phraser(data=data_to_code.text, 
                                  tokenizer=lambda x: mytokenizer.tokenize(x, stem=use_stemming, to_lowercase=to_lowercase), 
                                  min_score=min_score,
                                  show_df=False)
    text_to_code = tokens
else:
    text_to_code = data_to_code.text

text_transform = vectorizer.transform(text_to_code)
text_codes = classifier.predict(text_transform)
data_to_code = data_to_code.assign(topic=text_codes)
speech_data_coded = pd.concat([df_coded, data_to_code])

if save:
    if not os.path.isdir("data/"):
        os.mkdir("data/")
    pickle.dump(speech_data_coded, open("data/pp20_coded.pkl", "wb"))